Por brevedad, solo edito los datos y la salida de la segunda vuelta.

In [25]:
import pandas as pd
import numpy as np
import pyreadstat as pyd

In [26]:
# datos de la primera vuelta electoral
primeravuelta, meta1 = pyd.read_sav('Data/Segunda-Vuelta.sav', apply_value_formats=True)

In [27]:
primeravuelta.columns

Index(['DIGNIDAD_CODIGO', 'DIGNIDAD_NOMBRE', 'PROVINCIA_CODIGO',
       'PROVINCIA_NOMBRE', 'CIRCUNSCRIPCION_CODIGO', 'CIRCUNSCRIPCION_NOMBRE',
       'CANTON_CODIGO', 'CANTON_NOMBRE', 'PARROQUIA_CODIGO',
       'PARROQUIA_NOMBRE', 'JUNTA_SEXO', 'OP_CODIGO', 'CANDIDATO_NOMBRE',
       'SUFRAGANTES', 'BLANCOS', 'NULOS', 'VOTOS'],
      dtype='object')

In [28]:
# Ajustar tipo de datos - convertir float primero, luego int
primeravuelta = primeravuelta.astype({
    'DIGNIDAD_NOMBRE': str,
    'PROVINCIA_CODIGO': float,  # Primero a float
    'PROVINCIA_NOMBRE': str,
    'CIRCUNSCRIPCION_CODIGO': str,
    'CIRCUNSCRIPCION_NOMBRE': str,
    'CANTON_CODIGO': float,  # Primero a float
    'CANTON_NOMBRE': str, 
    'PARROQUIA_CODIGO': float,  # Primero a float
    'PARROQUIA_NOMBRE': str,
    'JUNTA_SEXO': str,
    'OP_CODIGO': float,  # Primero a float
    'SUFRAGANTES': float, 
    'BLANCOS': float,
    'NULOS': float, 
    'VOTOS': float,
    'CANDIDATO_NOMBRE': str
})

# Luego convertir a int (sin decimales)
primeravuelta[['PROVINCIA_CODIGO', 'CANTON_CODIGO', 'PARROQUIA_CODIGO', 'OP_CODIGO']] = \
primeravuelta[['PROVINCIA_CODIGO', 'CANTON_CODIGO', 'PARROQUIA_CODIGO', 'OP_CODIGO']].astype(int)

In [29]:
primeravuelta.head(5)

,DIGNIDAD_CODIGO,DIGNIDAD_NOMBRE,PROVINCIA_CODIGO,PROVINCIA_NOMBRE,CIRCUNSCRIPCION_CODIGO,CIRCUNSCRIPCION_NOMBRE,CANTON_CODIGO,CANTON_NOMBRE,PARROQUIA_CODIGO,PARROQUIA_NOMBRE,JUNTA_SEXO,OP_CODIGO,CANDIDATO_NOMBRE,SUFRAGANTES,BLANCOS,NULOS,VOTOS
0,11.0,PRESIDENTA/E Y VICEPRESIDENTA/E,1,AZUAY,0.0,,260,CUENCA,285,BAÑOS,FEMENINO,500,DANIEL NOBOA AZIN,9888.0,70.0,907.0,5563.0
1,11.0,PRESIDENTA/E Y VICEPRESIDENTA/E,1,AZUAY,0.0,,260,CUENCA,285,BAÑOS,FEMENINO,1644,LUISA GONZALEZ,9888.0,70.0,907.0,3346.0
2,11.0,PRESIDENTA/E Y VICEPRESIDENTA/E,1,AZUAY,0.0,,260,CUENCA,285,BAÑOS,MASCULINO,500,DANIEL NOBOA AZIN,8416.0,44.0,646.0,4808.0
3,11.0,PRESIDENTA/E Y VICEPRESIDENTA/E,1,AZUAY,0.0,,260,CUENCA,285,BAÑOS,MASCULINO,1644,LUISA GONZALEZ,8416.0,44.0,646.0,2917.0
4,11.0,PRESIDENTA/E Y VICEPRESIDENTA/E,1,AZUAY,0.0,,260,CUENCA,730,CUMBE,FEMENINO,500,DANIEL NOBOA AZIN,2402.0,19.0,202.0,1435.0


In [30]:
primeravuelta[['PROVINCIA_CODIGO', 'CANTON_CODIGO', 'PARROQUIA_CODIGO', 'PARROQUIA_NOMBRE']].head(5)

,PROVINCIA_CODIGO,CANTON_CODIGO,PARROQUIA_CODIGO,PARROQUIA_NOMBRE
0,1,260,285,BAÑOS
1,1,260,285,BAÑOS
2,1,260,285,BAÑOS
3,1,260,285,BAÑOS
4,1,260,730,CUMBE


In [31]:
dic = pd.read_excel('data/diccionario_parroquias.xlsx')

In [32]:
# Renombrar columnas para que coincidan con primeravuelta
dic = dic.rename(columns={'CODPRO': 'PROVINCIA_CODIGO',
                          'CODCAN': 'CANTON_CODIGO',
                          'CODPAR': 'PARROQUIA_CODIGO'}
                ).astype({'PROVINCIA_CODIGO': int, 
                          'CANTON_CODIGO': int,
                          'PARROQUIA_CODIGO': int
                          })

In [33]:
primeravuelta = primeravuelta.merge(
    dic,
    on = ['PROVINCIA_CODIGO', 'CANTON_CODIGO', 'PARROQUIA_CODIGO'], how='left'
    )

De momento, exploraremos solo los votos presidenciales

In [34]:
# Filtrar datos para PRESIDENTE Y VICEPRESIDENTE
presidente1 = primeravuelta[primeravuelta['DIGNIDAD_NOMBRE'] == 'PRESIDENTA/E Y VICEPRESIDENTA/E']

Los resultados de sufragantes, nulos y blancos solo pueden obtenerse a nivel de parroquia y junta de sexo. Además, hay que recordar que como el mapa del CNE es más viejo, hay que tratar la base para luego homologarla con el censo del 2022.

Observa el ejemplo siguiente.

In [35]:
# Así deben desagregarse los datos
presidente1[presidente1['ADM3_PCODE'] == 'EC010150']\
    .groupby(['ADM3_PCODE', 'PARROQUIA_NOMBRE', 'JUNTA_SEXO'], as_index=False)\
    .agg(Sufragantes=('SUFRAGANTES', 'mean'),
         Votos_válidos=('VOTOS', 'sum'),
         Nulos=('NULOS', 'mean'),
         Blancos=('BLANCOS', 'mean')
        )

,ADM3_PCODE,PARROQUIA_NOMBRE,JUNTA_SEXO,Sufragantes,Votos_válidos,Nulos,Blancos
0,EC010150,MACHANGARA,FEMENINO,4924.0,4618.0,288.0,18.0
1,EC010150,MACHANGARA,MASCULINO,4223.0,3924.0,284.0,15.0


In [36]:
# exportar para verificar sumas:
# presidente1[presidente1['PARROQUIA_NOMBRE'] == 'CAÑARIBAMBA'].to_excel("data/comprobaciones/canaribamba.xlsx", index=False)

Aun así hay leves descuadres en las sumas de válidos, blancos y nulos. Pero eso ya es cuestión de la fuente. De momento, vamos a tratar los datos como sigue:

In [37]:
# Antes de tener un diccionario homologado, se usó este filtro para eliminar parroquias problemáticas
# en los separados por sexo, pero ya no es necesario
resumen1 = presidente1.groupby(['ADM3_PCODE', 'JUNTA_SEXO'], as_index=False)\
    .agg(Sufragantes=('SUFRAGANTES', 'mean'),
         Votos_válidos=('VOTOS', 'sum'),
         Nulos=('NULOS', 'mean'),
         Blancos=('BLANCOS', 'mean')
         )

# Obtener resultados por parroquia
resumen1 = resumen1.groupby(['ADM3_PCODE'], as_index=False)\
    .agg(Sufragantes=('Sufragantes', 'sum'),
         Votos_válidos=('Votos_válidos', 'sum'),
         Nulos=('Nulos', 'sum'),
         Blancos=('Blancos', 'sum')
         )

resumen1.head(10)

,ADM3_PCODE,Sufragantes,Votos_válidos,Nulos,Blancos
0,EC010150,9147.0,8542.0,572.0,33.0
1,EC010151,18304.0,16634.0,1553.0,114.0
2,EC010152,4180.0,3837.0,309.0,35.0
3,EC010153,1024.0,912.0,88.0,24.0
4,EC010154,1662.0,1477.0,174.0,11.0
5,EC010155,2977.0,2646.0,309.0,22.0
6,EC010156,3225.0,2900.0,304.0,22.0
7,EC010157,4594.0,4168.0,385.0,40.0
8,EC010158,4364.0,4001.0,326.0,38.0
9,EC010159,1273.0,1117.0,137.0,19.0


In [38]:
# Calcular porcentajes de nulos y blancos
resumen1['Nulos_pct'] = resumen1['Nulos'] / resumen1['Sufragantes']
resumen1['Blancos_pct'] = resumen1['Blancos'] / resumen1['Sufragantes']
resumen1.head(10)

,ADM3_PCODE,Sufragantes,Votos_válidos,Nulos,Blancos,Nulos_pct,Blancos_pct
0,EC010150,9147.0,8542.0,572.0,33.0,0.062534,0.003608
1,EC010151,18304.0,16634.0,1553.0,114.0,0.084845,0.006228
2,EC010152,4180.0,3837.0,309.0,35.0,0.073923,0.008373
3,EC010153,1024.0,912.0,88.0,24.0,0.085938,0.023438
4,EC010154,1662.0,1477.0,174.0,11.0,0.104693,0.006619
5,EC010155,2977.0,2646.0,309.0,22.0,0.103796,0.007390
6,EC010156,3225.0,2900.0,304.0,22.0,0.094264,0.006822
7,EC010157,4594.0,4168.0,385.0,40.0,0.083805,0.008707
8,EC010158,4364.0,4001.0,326.0,38.0,0.074702,0.008708
9,EC010159,1273.0,1117.0,137.0,19.0,0.107620,0.014925


# Distribución de votos a candidatos por sufragante

In [39]:
# Agregamos clasificaciones ideológicas de candidatos
Political_Compass = pd.read_excel('data/PiliticalCompass.xlsx')
Political_Compass.head(3)

,Candidato,CANDIDATO_NOMBRE,Eje Económico (Izq–Der),Eje Social (Lib–Aut),Clasificación,Justificación breve,Separación
0,Luisa González (RC),LUISA GONZALEZ,–2.0,+1.5,Izquierda autoritaria,"Estado fuerte, redistribución, liderazgo centr...",LUISA GONZALEZ\n(Izquierda autoritaria)
1,Jorge Escala (Unidad Popular),JORGE ESCALA,–2.5,+2.0,Izquierda autoritaria,"Marxismo sindical, estatismo, orden desde movi...",Izquierda autoritaria
2,Pedro Granja (PSE),PEDRO GRANJA,–2.0,+1.0,Izquierda autoritaria,"Socialismo democrático radical, impuestos a gr...",Izquierda autoritaria


In [40]:
presidente1clas = presidente1.merge(Political_Compass[['CANDIDATO_NOMBRE', 'Separación']], on='CANDIDATO_NOMBRE', how='left')

In [41]:
# Distribución de votos por parroquia y por clasificación ideológica
distribucioncandidatos  = \
    presidente1clas.groupby(['ADM3_PCODE', 'Separación'], 
                            as_index=False)\
                                .agg(Sufragantes=('SUFRAGANTES', 'sum'), 
                                     Votos_candidato=('VOTOS', 'sum'))

In [42]:
distribucioncandidatos[distribucioncandidatos['Separación'] == 'Centro']

,ADM3_PCODE,Separación,Sufragantes,Votos_candidato


Existen descuadres en los resultados. Habría que considerarlo. (Valores que no suman en la tabla fuente).

In [43]:
distribucioncandidatos.head()
# Comprobar con distribucioncandidatos2.to_excel("data/comprobaciones/distribucioncandidatos_ec010150_sexo.xlsx", index=False)
# Recuerda filtrar un ADM3_PCODE

,ADM3_PCODE,Separación,Sufragantes,Votos_candidato
0,EC010150,DANIEL NOBOA AZIN\n(Derecha liberal \nmoderada),9147.0,5550.0
1,EC010150,LUISA GONZALEZ\n(Izquierda autoritaria),9147.0,2992.0
2,EC010151,DANIEL NOBOA AZIN\n(Derecha liberal \nmoderada),18304.0,10371.0
3,EC010151,LUISA GONZALEZ\n(Izquierda autoritaria),18304.0,6263.0
4,EC010152,DANIEL NOBOA AZIN\n(Derecha liberal \nmoderada),4180.0,2464.0


In [44]:
distribucioncandidatos['Votos_candidato_pct'] = distribucioncandidatos['Votos_candidato'] / distribucioncandidatos['Sufragantes']

In [45]:
distribucioncandidatos.head(4)

,ADM3_PCODE,Separación,Sufragantes,Votos_candidato,Votos_candidato_pct
0,EC010150,DANIEL NOBOA AZIN\n(Derecha liberal \nmoderada),9147.0,5550.0,0.606756
1,EC010150,LUISA GONZALEZ\n(Izquierda autoritaria),9147.0,2992.0,0.327102
2,EC010151,DANIEL NOBOA AZIN\n(Derecha liberal \nmoderada),18304.0,10371.0,0.566597
3,EC010151,LUISA GONZALEZ\n(Izquierda autoritaria),18304.0,6263.0,0.342166


In [46]:
reestructurado = distribucioncandidatos.\
    pivot(index=['ADM3_PCODE'], columns=['Separación'], 
          values='Votos_candidato_pct').fillna(0).reset_index()

In [47]:
# Unir tablas
table_resumen1 = resumen1.merge(reestructurado, on=['ADM3_PCODE'], how='left').fillna(0)

In [48]:
# Separamos por sexo y exportamos (porque será una estructura más manejable en Tableau)

# Verificar que no existan novedades
if distribucioncandidatos[distribucioncandidatos['Votos_candidato_pct'] > 1].empty:
    print("Todo en orden")
    table_resumen1.to_csv('data/2da.csv', index=False, decimal=',', sep=';')

Todo en orden
